<a href="https://colab.research.google.com/github/shamrosewebdev/End-to-End-ML-Pipeline-with-Scikit-learn-Pipeline-API/blob/main/End_to_End_ML_Pipeline_with_Scikit_learn_Pipeline_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
#Data loading
from sklearn.datasets import fetch_openml
import pandas as pd

# Fetch Telco Churn dataset
data = fetch_openml(name="Telco-Customer-Churn", version=1, as_frame=True)

df = data.frame
df.head()



,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,'No phone service',DSL,No,Yes,No,No,No,No,Month-to-month,Yes,'Electronic check',29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,'One year',No,'Mailed check',56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,'Mailed check',53.85,108.15,Yes
3,Male,0,No,No,45,No,'No phone service',DSL,Yes,No,Yes,Yes,No,No,'One year',No,'Bank transfer (automatic)',42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,'Fiber optic',No,No,No,No,No,No,Month-to-month,Yes,'Electronic check',70.70,151.65,Yes


In [30]:
# Target column
y = df["Churn"].map({"Yes": 1, "No": 0})

# Features
X = df.drop(columns=["Churn"])

In [31]:
# Identify numerical & categorical features

num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object"]).columns

print("Numerical Features:", list(num_features))
print("Categorical Features:", list(cat_features))


Numerical Features: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Categorical Features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']


In [32]:
# Build preprocessing pipelines

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])


In [33]:
# Categorical pipeline

from sklearn.preprocessing import OneHotEncoder

cat_pipeline = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [34]:
# Combine using ColumnTransformer

from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_features),
        ("cat", cat_pipeline, cat_features)
    ]
)


In [35]:
# Build full ML pipelines

# Logistic Regression Pipeline
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Random Forest pipeline
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])



In [36]:
# Hyperparameter tuning with GridSearchCV

# Logistic Regression grid
from sklearn.model_selection import GridSearchCV

logreg_params = {
    "classifier__C": [0.01, 0.1, 1, 10]
}

logreg_grid = GridSearchCV(
    logreg_pipeline,
    logreg_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


# Random Forest grid
rf_params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)




In [37]:
# Train the pipelines

logreg_grid.fit(X, y)
rf_grid.fit(X, y)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'Multip...
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'TotalCharges'],
      dtype='object'))])),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [None, 10, 20],
                         'classifier__n_estimators': [100, 200]},
             scoring='accuracy')

In [38]:
# Compare & select best model

print("Logistic Regression Best Accuracy:", logreg_grid.best_score_)
print("Random Forest Best Accuracy:", rf_grid.best_score_)

Logistic Regression Best Accuracy: 0.8036359200593587
Random Forest Best Accuracy: 0.7909987136266856


In [39]:
# Export the final pipeline

import joblib

best_model = rf_grid.best_estimator_
joblib.dump(best_model, "churn_pipeline.pkl")

# Model can be loaded and reused

model = joblib.load("churn_pipeline.pkl")
model.predict(X.iloc[:5])


array([0, 0, 1, 0, 1])